# Lanczos

给定厄密算符 $ H $ 计算它的基态

In [1]:
import quante as qt
import numpy as np

## 幂法

幂法：
$$
    H^{\Lambda} \left| \Psi  \right> \approx \left| \phi_{\text{ground}}  \right>
$$
当增加 ${ \Lambda }$，左边将接近右边

In [2]:
dim = 4000

# 生成随机的厄密矩阵
H = qt.generate.matrix.random_matrix(dim, type='herm') - 1 * np.eye(dim)
np.allclose(H, H.conj().T)

True

In [7]:
res = np.linalg.eigvalsh(H)
res[0]

-126.94309301556632

但实际需要的不是每个矩阵元，只要知道矩阵怎么作用到一个向量上就行了

In [3]:
# 获取矩阵作用到向量上的方法：
matvec = H.dot

v = np.random.randn(dim) + 1j * np.random.randn(dim)
v /= np.linalg.norm(v)

np.allclose(matvec(v), H @ v)

True

利用 matvec 反复作用就能得到基态

In [ ]:
Hv = v
for i in range(20):
    Hv = matvec(Hv)

# 可以近似基态能量
np.linalg.norm(Hv)**(1/20)  # exact: -126.94309301556632

111.63329887154897

增加 ${ \Gamma }$ 

In [ ]:
iter_num = 100

Hv = v
for i in range(iter_num):
    Hv = matvec(Hv)
    Hv /= np.linalg.norm(Hv)

# # 可以近似基态能量
np.linalg.norm(matvec(Hv))  # exact: -126.94309301556632

126.43370521072795

## Krylov 方法

为减少迭代的次数，构建由下面这组基矢张成的空间：
$$
    \left| \psi  \right>,  H \left| \psi  \right>,  H^{2} \left| \psi  \right>, \cdots , H^{\Lambda} \left| \psi  \right>
$$
但是这一组基矢不正交，不能构建矩阵，所以需要正交化

最简单的正交化方法是 QR 分解

In [20]:
bases = [v]
iter_num = 20
for i in range(iter_num-1):
    bases.append(matvec(bases[-1]))

# 现在需要正交化 bases 中的基矢
bases = np.array(bases).T
bases.shape

(4000, 20)

In [21]:
res = np.linalg.qr(bases)
res.Q.shape

(4000, 20)

证明 QR 就是正交化手续，要验证：
- Q 矩阵每列彼此正交
- bases 中的任何一个基矢能由 Q 中的各列叠加得到

In [22]:
# 验证正交性：
for i in range(res.Q.shape[0]):
    for j in range(i+1, res.Q.shape[1]):
        assert np.allclose(res.Q[:, i].conj() @ res.Q[:, j], 0)

# 验证可以叠加出 bases 中的任何一个基矢：
for i in range(bases.shape[1]):
    rebuilt_base = sum(res.R[j, i] * res.Q[:, j] for j in range(res.Q.shape[1]))
                # \sum_j     c_ij     newvec_j   == bases[i]
    assert np.allclose(rebuilt_base, bases[:, i])

有了新的基矢：
$$
    \left| \phi_1  \right>,  \left| \phi_2  \right>, \cdots , \left| \phi_{\Lambda}  \right>
$$
之后可以在这个子空间中写出厄密算符的矩阵（厄密算符在 Krylov 空间中的投影）
$$
    h_{ij} = \langle \phi_{i} | H | \phi_{j} \rangle
$$

In [23]:
# 填写矩阵元的方法：
h = np.zeros((iter_num, iter_num), dtype=complex)

for i in range(iter_num):
    for j in range(iter_num):
        h[i, j] = res.Q[:, i].conj() @ matvec(res.Q[:, j])

# 或者直接通过矩阵方法来得到
h_matmul = res.Q.conj().T @ matvec(res.Q)

np.allclose(h_matmul, h)

True

通过对角化这个矩阵，就可以得到基态能量更好的近似

In [ ]:
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 只乘了 20 次，精确到小数点后两位
# exact: -126.94309301556632

-126.03867054066637

QR 还是慢了，可以进一步加快运算：
通过：
$$
    \left| \psi_{m + 1}  \right> = H \left| \psi_{m}  \right> - a_{m} \left| \psi_{m}  \right> - b_{m - 1} \left| \psi_{m - 1}  \right>
$$
的方法构建基矢：
$$
    \left| \psi_{1}  \right>, \left| \psi_{2}  \right>, \cdots , \left| \psi_{\Lambda} \right>
$$
其中：
$$
    a_{m} = \langle \psi_{m} | H | \psi_{m} \rangle/\langle \psi_{m} | \psi_{m} \rangle ,\quad\;\; b_{m - 1} = \langle \psi_{m} | \psi_{m} \rangle/\langle \psi_{m - 1} | \psi_{m - 1} \rangle
$$
并且前两个基矢为：${ \left| \psi_2  \right> = H \left| \psi_{1}  \right> - a_1 \left| \psi_{1}  \right> }$ 

In [25]:
newbases = []

v1 = v/np.linalg.norm(v)
newbases.append(v1)

a1 = v1.conj().T @ matvec(v1)
v2 = matvec(v1) - a1 * v1
newbases.append(v2)

for i in range(2, iter_num):
    a2, b1 = (v2.conj().T @ matvec(v2)) / (v2.conj().T @ v2), (v2.conj().T @ v2)/(v1.conj().T @ v1)
    
    v3 = matvec(v2) - a2 * v2 - b1 * v1
    newbases.append(v3)
    v2, v1 = v3, v2

newbases = [i/np.linalg.norm(i) for i in newbases] # 归一化

In [26]:
# 验证正交性：
for i in range(len(newbases)):
    for j in range(i+1, len(newbases)):
        assert np.allclose(newbases[i].conj() @ newbases[j], 0)

# 因为基矢是由 H^n |psi> 叠加出的，所以必然在 Krylov 空间中：

In [27]:
newbases = np.array(newbases).T
newbases.shape

(4000, 20)

In [ ]:
h = newbases.conj().T @ matvec(newbases)
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 结果一样

-126.03867054065596

这时 h 是三对角的实矩阵：

In [29]:
h

array([[-1.54549923+0.j, 62.75076237-0.j, -0.        -0.j, -0.        -0.j,  0.        +0.j, -0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j, -0.        +0.j, -0.        -0.j, -0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j, -0.        -0.j,  0.        -0.j],
       [62.75076237+0.j, -1.20113501-0.j, 63.74295463-0.j,  0.        -0.j, -0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j,  0.        +0.j, -0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j, -0.        +0.j],
       [-0.        +0.j, 63.74295463+0.j,  0.59033526-0.j, 63.33763659+0.j,  0.        -0.j, -0.        +0.j,  0.        +0.j, -0.        +0.j,  0.        -0.j, -0.        +0.j,  0.        -0.j, -0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.        -0.j,  0.  

实际上可以直接生成这个矩阵：
$$
\begin{align*}
    \langle \psi_{m - 1} | H | \psi_{m} \rangle &= \sqrt{b_{m - 1}}\\
    \langle \psi_{m} | H | \psi_{m} \rangle &= a_m \\
    \langle \psi_{m + 1} | H | \psi_{m} \rangle &= \sqrt{b_{m}}\\
\end{align*}
$$

In [30]:
h = np.zeros((iter_num, iter_num), dtype=float)

v /= np.linalg.norm(v)
v1 = v

a1 = np.real(v.conj().T @ matvec(v))
h[0, 0] = a1

v2 = matvec(v) - a1 * v
b1 = np.real(v2.conj().T @ v2)
h[0, 1] = h[1, 0] = np.sqrt(b1)

for i in range(1, iter_num-1):
    Hv2 = matvec(v2)
    
    a2 = np.real(v2.conj().T @ Hv2  / (v2.conj().T @ v2))
    h[i, i] = a2
    
    v3 = Hv2 - a2 * v2 - b1 * v1 
    b2 = np.real(v3.conj().T @ v3 / (v2.conj().T @ v2))
    h[i, i+1] = h[i+1, i] = np.sqrt(b2)
    
    v2, v1 = v3, v2
    b1 = b2

Hv2 = matvec(v2)
a2 = np.real(v2.conj().T @ Hv2  / (v2.conj().T @ v2))
h[i+1, i+1] = a2

h  # 得到了完全一样的矩阵 （一共只做了 20 次矩阵向量乘法）

array([[-1.54549923, 62.75076237,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [62.75076237, -1.20113501, 63.74295463,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        , 63.74295463,  0.59033526, 63.33763659,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ],
       [ 0.        ,  0.        , 63.33763659, -0.98602759, 63.47497973,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.        ,  0.

In [31]:
res = np.linalg.eigh(h)
res.eigenvalues[0]  # 得到相同的结果

-126.03867054065614